In [64]:
import yfinance as yf
import pandas as pd
import pandas_datareader as pdr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

print("Libraries loaded successfully")

Libraries loaded successfully


# 1. Create The Dataset

#### 1.1 S&P500

In [53]:
start = datetime(2006, 1, 1)
end = datetime(2026, 7, 11)

In [75]:
sp500 = yf.Ticker("^GSPC")
sp500_data = sp500.history(start=start, end=end)["Close"].reset_index()
sp500_data.columns = ["Date", "SP500_Price"]

sp500_data["SP500_Return_5d"] = sp500_data["SP500_Price"].pct_change(5)
sp500_data["SP500_Return_21d"] = sp500_data["SP500_Price"].pct_change(21)
sp500_data["SP500_Return_63d"] = sp500_data["SP500_Price"].pct_change(63)

In [91]:
sp.reset_index()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,future_return,crash
0,2006-07-11 00:00:00-04:00,1267.260010,1273.640015,1259.650024,1272.430054,2310850000,0.0,0.0,0.061481,0
1,2006-07-12 00:00:00-04:00,1272.390015,1273.310059,1257.290039,1258.599976,2250450000,0.0,0.0,0.075338,0
2,2006-07-13 00:00:00-04:00,1258.579956,1258.579956,1241.430054,1242.280029,2545760000,0.0,0.0,0.086671,0
3,2006-07-14 00:00:00-04:00,1242.290039,1242.699951,1228.449951,1236.199951,2467120000,0.0,0.0,0.102435,0
4,2006-07-17 00:00:00-04:00,1236.199951,1240.069946,1231.489990,1234.489990,2146410000,0.0,0.0,0.106222,0
...,...,...,...,...,...,...,...,...,...,...
4963,2026-04-02 00:00:00-04:00,6512.609863,6601.910156,6474.939941,6582.689941,4740740000,0.0,0.0,0.145038,0
4964,2026-04-06 00:00:00-04:00,6587.660156,6618.129883,6579.720215,6611.830078,3906440000,0.0,0.0,0.134913,0
4965,2026-04-07 00:00:00-04:00,6601.930176,6618.259766,6534.549805,6616.850098,4555680000,0.0,0.0,0.130857,0
4966,2026-04-08 00:00:00-04:00,6754.359863,6793.500000,6740.279785,6782.810059,5904880000,0.0,0.0,0.112170,0


Gold and oil are classic crisis indicators :-

    - Gold rises during fear/uncertainty (safe haven)
    - Oil drops during economic slowdowns, spikes before inflation crashes

In [92]:
sp["future_return"] = sp["Close"].pct_change(63).shift(-63) 
sp["crash"] = (sp["future_return"] < -0.10 ).astype(int)
print(sp["crash"].value_counts())
print(sp["crash"].value_counts(normalize=True))

crash
0    4666
1     302
Name: count, dtype: int64
crash
0    0.939211
1    0.060789
Name: proportion, dtype: float64


In [93]:
sp = sp.dropna(subset=["future_return"])

302 day has been considered as a crash under our crash assumption , which are 10 months for over the last 20 year

the 6 major crash period:

- 2008 Financial Crisis
- 2011 European Debt Crisis
- 2015-2016 China slowdown
- 2018 Q4 correction
- 2020 COVID crash
- 2022 Rate hike crash

94% no crash
, 6% crash

the class are imbalanced , which means that we are going to use scale_pos_weight for tree models , or class balance = "Blanaced" for linear model

#### 1.2 Vix

popular measure of the stock market's expectation of volatility based on S&P 500 index options.

In [94]:
vix = pdr.get_data_fred("VIXCLS",start,end).reset_index()
vix.columns = ["Date","VIX"]

In [95]:
print(vix.head())
print(vix.shape)

        Date    VIX
0 2006-01-02    NaN
1 2006-01-03  11.14
2 2006-01-04  11.37
3 2006-01-05  11.31
4 2006-01-06  11.00
(5354, 2)


#### 1.3 Gold 

In [96]:
xau = yf.Ticker("GC=F")
gold = xau.history(start=start, end=end)["Close"].reset_index()
gold.columns = ["Date", "Gold_Price"]

gold["Gold_Return_63d"] = gold["Gold_Price"].pct_change(63)

gold["Gold_Return_21d"] = gold["Gold_Price"].pct_change(21)
gold["Gold_Return_5d"] = gold["Gold_Price"].pct_change(5)

#### 1.4 Oil

In [97]:
oil = yf.Ticker("CL=F")
oil_data = oil.history(start=start, end=end)["Close"].reset_index()
oil_data.columns = ["Date", "Oil_Price"]

oil_data["Oil_Return_63d"] = oil_data["Oil_Price"].pct_change(63)
oil_data["Oil_Return_21d"] = oil_data["Oil_Price"].pct_change(21)
oil_data["Oil_Return_5d"] = oil_data["Oil_Price"].pct_change(5)

#### 1.5 Yield Curve

In [98]:
t10 = pdr.get_data_fred("GS10",start,end).reset_index()
t10.columns = ["Date","T10Y"]

t2 = pdr.get_data_fred("GS2",start,end).reset_index()
t2.columns = ("Date","T2Y")

#### 1.6 FED Rate

In [99]:
fed = pdr.get_data_fred("FEDFUNDS", start, end).reset_index()
fed.columns = ["Date", "Fed_Rate"]

#### 1.7 CPI

In [100]:
cpi = pdr.get_data_fred("CPIAUCSL", start, end).reset_index()
cpi.columns = ["Date", "CPI"]

#### 1.8 Unemployment

In [101]:
unemp = pdr.get_data_fred("UNRATE", start, end).reset_index()
unemp.columns = ["Date", "Unemployment"]

In [102]:
print(t10.shape, t2.shape, fed.shape, cpi.shape, unemp.shape)


(246, 2) (246, 2) (246, 2) (245, 2) (246, 2)


#### 1.9 Merge 

In [104]:
sp = sp.reset_index()

In [105]:
for df in [sp,vix,gold,oil_data,t10,t2,fed,cpi,unemp]:
    df["Date"] = pd.to_datetime(df["Date"]).dt.tz_localize(None) if df["Date"].dt.tz is not None else df["Date"]
    df.sort_values("Date", inplace=True)

In [107]:
master = sp.copy()

In [108]:
for df, name in [(vix, "vix"), (gold, "gold"), (oil_data, "oil"),
                  (t10, "t10"), (t2, "t2"), (fed, "fed"),
                  (cpi, "cpi"), (unemp, "unemp")]:
    master = pd.merge_asof(master, df, on="Date", direction="backward")

print(master.shape)
print(master.isnull().sum())

(4905, 24)
Date                0
Open                0
High                0
Low                 0
Close               0
Volume              0
Dividends           0
Stock Splits        0
future_return       0
crash               0
VIX                 0
Gold_Price          0
Gold_Return_63d     0
Gold_Return_21d     0
Gold_Return_5d      0
Oil_Price           0
Oil_Return_63d      0
Oil_Return_21d      0
Oil_Return_5d       0
T10Y                0
T2Y                 0
Fed_Rate            0
CPI                23
Unemployment       23
dtype: int64


In [109]:
master = master.dropna(subset=["CPI", "Unemployment"])
master["Yield_Spread"] = master["T10Y"] - master["T2Y"]
print(master.shape)

(4882, 25)
